In [ ]:

%load_ext sql
%reload_ext sql
%config SqlMagic.displaylimit = 10
%sql sqlite:///../data/videos.db
%config SqlMagic.feedback = 1

%load_ext autoreload
%autoreload 2

# add to path
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

import pandas as pd

pd.set_option('display.width', 1024)
pd.set_option('display.max_colwidth', 256)

In [ ]:
videos_data = %sql SELECT * FROM videos;
videos_df = videos_data.DataFrame()
# drop na in statistics_viewCount
# videos_df = videos_df.dropna(subset=['statistics_viewCount'])
# drop if statistics_viewCount is NaN
videos_df = videos_df[~videos_df['statistics_viewCount'].isnull()]
videos_df['statistics_viewCount'] = videos_df['statistics_viewCount'].astype(int)
videos_df

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

titles = videos_df['snippet_title']

# Stemmize the titles
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
stemmed_titles = titles.apply(lambda x: ' '.join([stemmer.stem(word) for word in x.split()]))

import numpy as np

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(stemmed_titles)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out(), index=videos_df['id'])
tfidf_df

KeyboardInterrupt: 

In [ ]:
from sklearn.model_selection import train_test_split

X = tfidf_df.copy()
y = videos_df['statistics_viewCount'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV

lasso = Lasso(max_iter=10000)
params = {'alpha': [0.1, 1, 10, 100]}
lasso_regressor = GridSearchCV(lasso, params, cv=2)
lasso_regressor.fit(X_train, y_train)
lasso_model = lasso_regressor.best_estimator_

print(f'Lasso score: {lasso_model.score(X_test, y_test)}')

# print coefficients
lasso_coefficients = pd.DataFrame(lasso_model.coef_, index=tfidf_df.columns, columns=['Coefficient'])
lasso_coefficients.sort_values('Coefficient', ascending=False, inplace=True)

from xgboost import XGBRegressor

xgb = XGBRegressor()
params = {'n_estimators': [100, 500, 1000], 'max_depth': [3, 5, 7]}
xgb_regressor = GridSearchCV(xgb, params, cv=2)
xgb_regressor.fit(X_train, y_train)
xgb_model = xgb_regressor.best_estimator_

print(f'XGBoost score: {xgb_model.score(X_test, y_test)}')

# print feature importances
xgb_importances = pd.DataFrame(xgb_model.feature_importances_, index=tfidf_df.columns, columns=['Importance'])
xgb_importances.sort_values('Importance', ascending=False, inplace=True)

In [28]:
features = pd.concat([lasso_coefficients, xgb_importances], axis=1)
features = features[features['Importance'] * features['Coefficient'] != 0]
features

,Coefficient,Importance
000,5.739448e+04,0.001626
10,3.000250e+05,0.002994
1080p,-5.690556e+05,0.004832
11,-2.119754e+04,0.001481
13,-4.089187e+04,0.001950
...,...,...
week,-4.826206e+05,0.001207
wheel,1.672815e+06,0.012798
world,-6.042935e+04,0.001485
year,-2.650883e+04,0.001552


In [29]:
# train keras seqential model
from keras.models import Sequential
from keras.layers import Dense
from tensorflow.keras import backend as K

model_X_train = X_train.copy()
model_X_test = X_test.copy()

model_X_train = model_X_train[features.index]
model_X_test = model_X_test[features.index]

def keras_r2(y_true, y_pred):
    # convert to keras float
    y_true = K.cast(y_true, K.floatx())
    y_pred = K.cast(y_pred, K.floatx())
    SS_res =  K.sum(K.square(y_true - y_pred)) 
    SS_tot = K.sum(K.square(y_true - K.mean(y_true))) 
    return (1 - SS_res/(SS_tot + K.epsilon()))

model = Sequential()
model.add(Dense(1024, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(1, activation='linear'))

model.compile(loss='mean_squared_error', optimizer='adam', metrics=[keras_r2])
model.fit(X_train, np.log1p(y_train), epochs=10, batch_size=32, validation_split=0.2)

_, keras_score = model.evaluate(X_test, np.log1p(y_test))
print(f'Keras score: {keras_score}')

c:\Users\Mitchell\Projects\Tools\ThumbnailModel\.conda\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
3903/3903 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - keras_r2: -1.9672 - loss: 10.6331 - val_keras_r2: 0.1950 - val_loss: 3.1713
Epoch 2/10
3903/3903 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - keras_r2: 0.2303 - loss: 2.9491 - val_keras_r2: 0.2586 - val_loss: 2.9134
Epoch 3/10
3903/3903 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - keras_r2: 0.3096 - loss: 2.6629 - val_keras_r2: 0.2837 - val_loss: 2.8128
Epoch 4/10
3903/3903 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - keras_r2: 0.3601 - loss: 2.4705 - val_keras_r2: 0.2511 - val_loss: 2.9061
Epoch 5/10
3903/3903 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - keras_r2: 0.3904 - loss: 2.3109 - val_keras_r2: 0.2599 - val_loss: 2.8688
Epoch 6/10
3903/3903 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - keras_r2: 0.4282 - loss: 2.1923 - val_keras_r2: 0.2812 - val_loss: 2.7921
Epoch 7/10
3903/3903 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - keras_r2: 0.4455 - loss: 2.1134 - val_keras_r2: 0.2857 - val_loss: 2.7692
Epoch 8/10
3903/3903 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - keras_r2: 0.4645 - loss: 2.0267 

1220/1220 ━━━━━━━━━━━━━━━━━━━━ 1s 616us/step - keras_r2: 0.2619 - loss: 2.8220
Keras score: 0.26552730798721313
